In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import numpy as np
import re
import pandas as pd
import itables
from itables import show
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag, compare
from src.evaluation import process_matching_row, apply_matching_processing

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

In [ ]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Searching is starting in directory: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"{len(all_pairs)} JSON/MD file pairs were found in total.")
else:
    logger.error("The DATA_DIR environment variable was not found in the .env file!")

In [ ]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"The merging is completed! A total of {len(main_dataset)} question-answer pairs were found!")

In [ ]:
subject_translation = {
    "nea_ellinika": "greek_language",
    "arxaia": "ancient_greek",
    "istoria": "history",
    "latinika": "latin",
    "biologia": "biology",
    "fysiki": "physics",
    "ximeia": "chemistry",
    "pliroforiki": "computer_science",
    "arxes_oikonomikis_theorias": "economics",
    "mathimatika": "mathematics"
}

In [ ]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    old_subj = item.get("subject", "")
    new_subj = subject_translation.get(old_subj, old_subj)
    item["subject"] = new_subj
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)
    
    year = item.get("year", "")
    old_id = item.get("id", "")
    school_type = str(item.get("school_type", "gel")).lower()
    item["id"] = f"{new_subj}_{school_type}_{year}_{old_id}"

In [ ]:
images_found = 0
print("--- Questions that were found to have images ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} in subject {item.get('subject')} ({item.get('year')}) - Contains {len(imgs)} image(s)")

print(f"\nA total of {images_found} questions (IDs) with images were found.")

In [ ]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "panellinies_dataset.xlsx"

In [ ]:
df = pd.DataFrame(main_dataset)
df = df.rename (columns={"answer": "answer_text"})
my_columns = [
    "id",
    "subject",
    "format",
    "reference",
    "question",
    "input",
    "images",
    "choices",
    "answer_text",
    "answer_index",
    "image_description",
    "image_transcription",
    "points",
    "year",
    "school_type"
]

df = df[my_columns]

In [ ]:
df.to_excel(output_file, index=False)

logger.info (f"The file was successfully created at: {output_file.resolve()}!")

df.head()

In [ ]:
reference_file = Path("../results/panellinies_dataset.xlsx")

compare_results = compare(df, reference_file)

if isinstance (compare_results, dict):
    print("\n--- IDs that exist ONLY in the new dataset ---")
    display(compare_results["only_current"][["id"]].head(20))
    
    print("\n--- IDs that exist ONLY in the reference dataset ---")
    display(compare_results["only_reference"][["id"]].head(20))

In [ ]:
#testing conversion of matching questions into multiple choice ones for LLM evaluation
test_df = apply_matching_processing(df)

matching_only_df = test_df[test_df['format'] == 'matching'][['id', 'processed_choices', 'new_answer_index']]

show(matching_only_df,
     layout={"top1": "searchBuilder"},
     buttons=[
         "pageLength",
         {"extend": "excelHtml5", "title": "test_df_matching_questions"}
     ]
)

In [ ]:
#matching_questions validation
first_matching = test_df[test_df['format'] == 'matching'].iloc[0]

print(f"Number of options: {len(first_matching['processed_choices'])}")
print(f"The 1st option is: {first_matching['processed_choices'][0]}")